# Fetch Iowa Liquor Sales raw data

Downloads the Iowa Liquor Sales raw data from the official Iowa Data Hub API and records where the project data comes from.

**Source datasets** (Iowa Data Hub, Department of Revenue, licence CC BY 4.0)

| Year | Dataset ID | Coverage | Dataset page |
|---|---|---|---|
| 2025 | 1262 | 2025-01-01 to 2025-12-31 | https://data.iowa.gov/catalog/dataset/1262 |
| 2026 | 1263 | From 2026-01-01, updated monthly | https://data.iowa.gov/catalog/dataset/1263 |

**Frozen release used in this project**: published 2026-10-06, orders from 2025-01-01 to 2026-09-30, with 2,490,620 rows for 2025 and 1,814,856 rows for 2026.

All analysis uses this frozen snapshot. Running this notebook again downloads the release that is live on the portal at that time, and the checks below flag any difference from the frozen release.

**How to run**: set `YEARS` and `OUT_DIR` in the Settings cell, then run all cells.

**Output**
- `<OUT_DIR>/snapshot_<YYYYMMDD>/`: the downloaded file (the API returns a ZIP archive of CSV parts) and the extracted CSV parts
- `<OUT_DIR>/manifest.json`: download time, source URL, size, SHA-256, and rows per CSV part

Requires Python 3.9+ and `certifi` for HTTPS verification (included in Anaconda; otherwise run `%pip install certifi`).

In [13]:
import codecs
import csv
import hashlib
import json
import shutil
import ssl
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path

BASE_URL = "https://idh-be.iowa.gov/api/v1/datasets/{dataset_id}/rows.csv"

# The Iowa Data Hub is served through Cloudflare with a Let's Encrypt certificate whose chain
# ends in an ISRG Root X2 cross-signature that expired on 2025-09-15. A trust store without
# the self-signed ISRG Root X2 (often the case for the Windows store that Python reads) then
# reports "CERTIFICATE_VERIFY_FAILED: certificate has expired". The certifi CA bundle includes
# ISRG Root X2, so HTTPS verification uses certifi.
try:
    import certifi
except ImportError:
    raise ImportError(
        "certifi is required. Run  %pip install certifi  in a cell, restart the kernel, then rerun."
    ) from None

SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())
print("HTTPS verification: certifi CA bundle", certifi.__version__)

HTTPS verification: certifi CA bundle 2024.12.14


In [14]:
# Dataset IDs and row counts of the frozen release (published 2026-10-06)
DATASETS = {
    "2025": {"dataset_id": 1262, "expected_rows": 2_490_620},
    "2026": {"dataset_id": 1263, "expected_rows": 1_814_856},
}

In [15]:
# Column order of the frozen release, matching the MySQL table iowa_liquor_sales
EXPECTED_COLUMNS = [
    "invoice_id", "ordered_on", "store_no", "store_name", "store_address",
    "store_city", "store_zip_code", "county_fips_code", "county_name",
    "category_code", "category_name", "vendor_number", "vendor_name",
    "item_no", "im_desc", "pack", "bottle_volume_ml", "state_bottle_cost",
    "state_bottle_retail", "sales_bottles", "sales_dollars", "sales_liters",
    "sales_gallons",
]

CHUNK_SIZE = 1024 * 1024  # 1 MiB

In [16]:
def download(url: str, dest: Path) -> tuple[int, str]:
    """Stream the file at url to dest. Return (bytes written, SHA-256 hex digest)."""
    sha = hashlib.sha256()
    size = 0
    request = urllib.request.Request(url, headers={"User-Agent": "pal-data-fetch/1.0"})
    with urllib.request.urlopen(request, timeout=600, context=SSL_CONTEXT) as response, open(dest, "wb") as f:
        while chunk := response.read(CHUNK_SIZE):
            f.write(chunk)
            sha.update(chunk)
            size += len(chunk)
    return size, sha.hexdigest()


def unpack(download_path: Path, snapshot_dir: Path, stem: str) -> tuple[Path, list[Path]]:
    """The API returns a ZIP archive of CSV parts; a plain CSV is also accepted.
    Return (downloaded file, CSV files to check)."""
    if zipfile.is_zipfile(download_path):
        zip_path = download_path.replace(snapshot_dir / f"{stem}.zip")
        csv_paths = []
        with zipfile.ZipFile(zip_path) as zf:
            for member in sorted(m for m in zf.namelist() if m.lower().endswith(".csv")):
                target = snapshot_dir / Path(member).name  # file name only, no folders from the archive
                with zf.open(member) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst, CHUNK_SIZE)
                csv_paths.append(target)
        return zip_path, csv_paths
    csv_path = download_path.replace(snapshot_dir / f"{stem}.csv")
    return csv_path, [csv_path]


def is_utf8(path: Path) -> bool:
    """True if the whole file decodes as UTF-8, the encoding the MySQL load script expects."""
    decoder = codecs.getincrementaldecoder("utf-8")()
    with open(path, "rb") as f:
        try:
            while chunk := f.read(CHUNK_SIZE):
                decoder.decode(chunk)
            decoder.decode(b"", final=True)
        except UnicodeDecodeError:
            return False
    return True


def inspect_csv(path: Path) -> tuple[bool, list[str], int]:
    """Return (starts with the header row, first row, number of data rows).
    Parses the CSV so a quoted line break counts once."""
    with open(path, newline="", encoding="utf-8-sig", errors="replace") as f:
        reader = csv.reader(f)
        first = next(reader, [])
        has_header = first == EXPECTED_COLUMNS
        n_rows = sum(1 for _ in reader) + (0 if has_header or not first else 1)
    return has_header, first, n_rows


def fetch(years: list[str], out_dir: Path) -> bool:
    """Download the selected years, unpack the CSV parts, check them against the frozen release,
    and append one record per year to manifest.json. Return True when every check passes."""
    unknown = [y for y in years if y not in DATASETS]
    if unknown:
        raise ValueError(f"Unknown year(s) {unknown}; choose from {list(DATASETS)}")

    out_dir = Path(out_dir)
    started = datetime.now(timezone.utc)
    snapshot_dir = out_dir / f"snapshot_{started:%Y%m%d}"
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else []
    all_checks_passed = True

    for year in years:
        cfg = DATASETS[year]
        url = BASE_URL.format(dataset_id=cfg["dataset_id"])
        stem = f"iowa_liquor_sales_{year}_{cfg['dataset_id']}"

        print(f"[{year}] downloading {url}")
        size, sha256 = download(url, snapshot_dir / f"{stem}.download")
        source_path, csv_paths = unpack(snapshot_dir / f"{stem}.download", snapshot_dir, stem)
        print(f"[{year}] saved {source_path} | {size / 1e6:,.1f} MB | {len(csv_paths)} CSV file(s)")

        parts, total_rows = [], 0
        for i, path in enumerate(csv_paths):
            has_header, first_row, n_rows = inspect_csv(path)
            utf8 = is_utf8(path)
            total_rows += n_rows
            parts.append({"file": path.name, "rows": n_rows, "has_header": has_header, "utf8": utf8})
            print(f"    {path.name}: {n_rows:,} rows | header row: {'yes' if has_header else 'no'}")
            if i == 0 and not has_header:
                print(f"    WARNING: first file does not start with the expected columns: {first_row[:5]} ...")
            if not utf8:
                print("    WARNING: file is not valid UTF-8; set the matching CHARACTER SET in LOAD DATA")

        columns_match = bool(parts) and parts[0]["has_header"]
        rows_match = total_rows == cfg["expected_rows"]
        encoding_ok = all(p["utf8"] for p in parts)
        if not rows_match:
            print(
                f"[{year}] WARNING: {total_rows:,} rows, frozen release has {cfg['expected_rows']:,}. "
                "The portal may have published a newer release, or the download is incomplete."
            )
        passed = columns_match and rows_match and encoding_ok
        all_checks_passed = all_checks_passed and passed
        print(f"[{year}] total {total_rows:,} rows | {'matches' if passed else 'differs from'} the frozen release")

        manifest.append({
            "year": year,
            "dataset_id": cfg["dataset_id"],
            "source_url": url,
            "downloaded_at_utc": started.isoformat(timespec="seconds"),
            "download_file": f"{snapshot_dir.name}/{source_path.name}",
            "bytes": size,
            "sha256": sha256,
            "csv_parts": parts,
            "rows": total_rows,
            "matches_frozen_release": passed,
        })

    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"manifest written to {manifest_path}")
    return all_checks_passed

## Settings

Running the next cells downloads about 4.3 million rows from the portal. Set `YEARS = ["2026"]` to download one year only.

In [17]:
YEARS = ["2025", "2026"]
OUT_DIR = Path("data/api_download")  # kept apart from the frozen files in "data/raw data"

## Download and check

In [18]:
passed = fetch(YEARS, OUT_DIR)
print("All checks passed." if passed else "Some checks failed. See the warnings above.")

[2025] downloading https://idh-be.iowa.gov/api/v1/datasets/1262/rows.csv
[2025] saved data\api_download\snapshot_20261016\iowa_liquor_sales_2025_1262.zip | 631.3 MB | 5 CSV file(s)
    iowa_liquor_sales_2025_1262_rows_part_0001.csv: 598,080 rows | header row: yes
    iowa_liquor_sales_2025_1262_rows_part_0002.csv: 775,752 rows | header row: yes
    iowa_liquor_sales_2025_1262_rows_part_0003.csv: 628,867 rows | header row: yes
    iowa_liquor_sales_2025_1262_rows_part_0004.csv: 487,921 rows | header row: yes
    iowa_liquor_sales_2025_1262_rows_part_0005.csv: 429,989 rows | header row: yes
[2025] WARNING: 2,920,609 rows, frozen release has 2,490,620. The portal may have published a newer release, or the download is incomplete.
[2025] total 2,920,609 rows | differs from the frozen release
[2026] downloading https://idh-be.iowa.gov/api/v1/datasets/1263/rows.csv
[2026] saved data\api_download\snapshot_20261016\iowa_liquor_sales_2026_1263.zip | 399.3 MB | 4 CSV file(s)
    iowa_liquor_sales